In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data path:", DATA_RAW.resolve())
print("Processed data path:", DATA_PROCESSED.resolve())

Project root: /workspaces/potential-pathway-index
Raw data path: /workspaces/potential-pathway-index/data/raw
Processed data path: /workspaces/potential-pathway-index/data/processed


In [4]:
required_files = {
    "base_student_dataset": DATA_PROCESSED / "base_student_dataset.csv",
    "student_vle": DATA_RAW / "studentVle.csv",
    "student_assessment": DATA_RAW / "studentAssessment.csv",
    "assessments": DATA_RAW / "assessments.csv"
}

for file_label, file_path in required_files.items():
    print(f"{file_label:25} | exists: {file_path.exists()} | path: {file_path}")

base_student_dataset      | exists: True | path: ../data/processed/base_student_dataset.csv
student_vle               | exists: True | path: ../data/raw/studentVle.csv
student_assessment        | exists: True | path: ../data/raw/studentAssessment.csv
assessments               | exists: True | path: ../data/raw/assessments.csv


In [5]:
base_student_dataset = pd.read_csv(required_files["base_student_dataset"])

print("base_student_dataset loaded successfully.")
print("Shape:", base_student_dataset.shape)

base_student_dataset.head()

base_student_dataset loaded successfully.
Shape: (32593, 14)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,final_result,at_risk_misalignment
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,Pass,0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,Pass,0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,Withdrawn,1
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,Pass,0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,-176.0,Pass,0


In [6]:
student_vle_columns = [
    "code_module",
    "code_presentation",
    "id_student",
    "date",
    "sum_click"
]

student_vle = pd.read_csv(
    required_files["student_vle"],
    usecols=student_vle_columns
)

print("student_vle loaded successfully.")
print("Shape:", student_vle.shape)

student_vle.head()

student_vle loaded successfully.
Shape: (10655280, 5)


,code_module,code_presentation,id_student,date,sum_click
0,AAA,2013J,28400,-10,4
1,AAA,2013J,28400,-10,1
2,AAA,2013J,28400,-10,1
3,AAA,2013J,28400,-10,11
4,AAA,2013J,28400,-10,1


In [7]:
student_assessment = pd.read_csv(required_files["student_assessment"])

assessments = pd.read_csv(required_files["assessments"])

print("student_assessment loaded successfully.")
print("Shape:", student_assessment.shape)

print("\nassessments loaded successfully.")
print("Shape:", assessments.shape)

student_assessment loaded successfully.
Shape: (173912, 5)

assessments loaded successfully.
Shape: (206, 6)


In [8]:
main_keys = ["id_student", "code_module", "code_presentation"]

EARLY_WINDOW_START = 0
EARLY_WINDOW_END = 30

print("Main keys:", main_keys)
print("Early observation window:", EARLY_WINDOW_START, "to", EARLY_WINDOW_END)

Main keys: ['id_student', 'code_module', 'code_presentation']
Early observation window: 0 to 30


In [9]:
base_total_rows = base_student_dataset.shape[0]
base_unique_keys = base_student_dataset[main_keys].drop_duplicates().shape[0]

print("base_student_dataset total rows:", base_total_rows)
print("base_student_dataset unique student-module-presentation keys:", base_unique_keys)
print("Is the main key unique in base_student_dataset?", base_total_rows == base_unique_keys)

base_student_dataset total rows: 32593
base_student_dataset unique student-module-presentation keys: 32593
Is the main key unique in base_student_dataset? True


In [10]:
print("student_vle date minimum:", student_vle["date"].min())
print("student_vle date maximum:", student_vle["date"].max())

print("\nNumber of rows before course start:")
print((student_vle["date"] < 0).sum())

print("\nNumber of rows inside early window:")
print(student_vle["date"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of rows after early window:")
print((student_vle["date"] > EARLY_WINDOW_END).sum())

student_vle date minimum: -25
student_vle date maximum: 269

Number of rows before course start:
688988

Number of rows inside early window:
2291587

Number of rows after early window:
7674705


In [11]:
print("student_assessment date_submitted minimum:", student_assessment["date_submitted"].min())
print("student_assessment date_submitted maximum:", student_assessment["date_submitted"].max())

print("\nNumber of submissions before course start:")
print((student_assessment["date_submitted"] < 0).sum())

print("\nNumber of submissions inside early window:")
print(student_assessment["date_submitted"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of submissions after early window:")
print((student_assessment["date_submitted"] > EARLY_WINDOW_END).sum())

student_assessment date_submitted minimum: -11
student_assessment date_submitted maximum: 608

Number of submissions before course start:
2057

Number of submissions inside early window:
24465

Number of submissions after early window:
147390


In [12]:
print("assessments due date minimum:", assessments["date"].min())
print("assessments due date maximum:", assessments["date"].max())

print("\nMissing assessment due dates:")
print(assessments["date"].isnull().sum())

print("\nAssessment types:")
print(assessments["assessment_type"].value_counts())

assessments due date minimum: 12.0
assessments due date maximum: 261.0

Missing assessment due dates:
11

Assessment types:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64
